In [ ]:
from Declare4Py.ProcessModels.DeclareModel import DeclareModel
from Declare4Py.ProcessMiningTasks.Discovery.DeclareMiner import DeclareMiner
from Declare4Py.D4PyEventLog import D4PyEventLog
from Declare4Py.ProcessModels.DeclareModel import DeclareModelTemplate

In [ ]:
event_log_name = "p2p"
log_path = f"D:\\LTNcoder\\.out\\eventlogs\\{event_log_name}-0.3-1.xes"

In [ ]:
event_log = D4PyEventLog(case_name="case:concept:name")
event_log.parse_xes_log(log_path)

In [ ]:
discovery = DeclareMiner(log=event_log, consider_vacuity=False, min_support=0.05, itemsets_support=0.05, max_declare_cardinality=1)
declare_model: DeclareModel = discovery.run()
print(f"Total constraints discovered: {len(declare_model.serialized_constraints)}")
model_constraints = declare_model.get_decl_model_constraints()
# print("Model constraints:")
# print("-----------------")
# for idx, constr in enumerate(model_constraints):
#     print(idx, constr)


In [ ]:
# declare_model.to_file(f"D:\\LTNcoder\\.out\\decl\\{event_log_name}-0.3-1.decl")
declare_model.to_file(f"{event_log_name}-0.3-1.decl")


In [ ]:
from Declare4Py.ProcessMiningTasks.ConformanceChecking.MPDeclareAnalyzer import MPDeclareAnalyzer
from Declare4Py.ProcessMiningTasks.ConformanceChecking.MPDeclareResultsBrowser import MPDeclareResultsBrowser

basic_checker = MPDeclareAnalyzer(log=event_log, declare_model=declare_model, consider_vacuity=False)
conf_check_res: MPDeclareResultsBrowser = basic_checker.run()

In [ ]:
import pickle

# Save the conformance checking results to disk to avoid recalculating
def save_conformance_results(conf_check_res, filename=f'{event_log_name}_5000_conformance_results.pkl'):
    with open(filename, 'wb') as f:
        pickle.dump(conf_check_res, f)
    print(f"Conformance checking results saved to {filename}")

# Load the conformance checking results from disk
def load_conformance_results(filename=f'{event_log_name}_5000_conformance_results.pkl'):
    try:
        with open(filename, 'rb') as f:
            conf_check_res = pickle.load(f)
        print(f"Conformance checking results loaded from {filename}")
        return conf_check_res
    except FileNotFoundError:
        print(f"File {filename} not found. Run conformance checking first.")
        return None



In [ ]:
# Save the current conformance results
save_conformance_results(conf_check_res)

In [ ]:
conf_check_res = load_conformance_results()


In [ ]:
conf_check_df =  conf_check_res.get_metric(metric="state")
# display(conf_check_df)

In [ ]:
summary_df = conf_check_df.apply(lambda col: col.value_counts()).fillna(0).astype(int)
summary_df = summary_df.reindex([0, 1])
summary_df = summary_df / len(conf_check_df)
summary_df = summary_df.T
summary_df = summary_df.sort_values(by=1, ascending=False)
display(summary_df)

In [ ]:
import pandas as pd

In [ ]:
activated_df = conf_check_res.get_metric(metric="num_activations").fillna(0)
satisfied_df = conf_check_res.get_metric(metric="state").fillna(0)
import pandas as pd
import numpy as np

# Support = how often constraint is satisfied
support = satisfied_df.sum(axis=0) / len(satisfied_df)

# Activation rate = how often constraint is activated
activated_counts = activated_df.sum(axis=0)
activation_rate = activated_counts / len(activated_df)

# Satisfied counts
satisfied_counts = satisfied_df.sum(axis=0)

# Confidence = P(satisfied | activated), safe division
confidence = np.where(
    activated_counts != 0,
    satisfied_counts / activated_counts,
    0
)
confidence = pd.Series(confidence, index=activated_counts.index)

# Combine into a DataFrame
metrics_df = pd.DataFrame({
    'support': support,
    'confidence': confidence,
    # 'activation_rate': activation_rate
})

In [ ]:
metrics_df

In [ ]:
# filtered_metrics_df is metrics_df but with rows with support less than 0.2 and in descending order of confidence and no "not" in the constraint name
filtered_metrics_df = metrics_df[~metrics_df.index.str.contains("Not") & (metrics_df['support'] <= 0.2) & (metrics_df['confidence'] >= 0.7)].sort_values(by='confidence', ascending=False)
# filtered_metrics_df = metrics_df[metrics_df['support'] <= 0.2].sort_values(by='confidence', ascending=False)
print("Filtered Metrics DataFrame:")
display(filtered_metrics_df)

# Constraints with low support and high confidence
1. Responded Existence[Create PR, Pay] | |	0.1534	0.9948119325551232	0.1542
2. Responded Existence[Approve PO 2, Post IR] | |	0.0756	0.9921259842519685	0.0762
3. Response[Approve PO 2, Pay] | |	0.0756	0.9921259842519685	0.0762
4. Response[Approve PO 2, Release PO] | |	0.073	0.958005249343832	0.0762 
5. *Response[Release PR, Pay] | |	0.1536	0.9858793324775353*

In [ ]:
interesting_constraints = [
    # "Response[Approve PO 2, Release PO] | |",
    "Response[Release PR, Pay] | |"
    ]

In [ ]:
state_df = conf_check_res.get_metric("state")[interesting_constraints]
non_zero_counts = state_df.ne(0).sum(axis=0)
print(non_zero_counts)
non_zero_rows = state_df.index[state_df[interesting_constraints[0]] != 0].tolist()
print(non_zero_rows)
print(len(non_zero_rows))
with open(f'{event_log_name}_ltn_rows.pkl', 'wb') as f:
    pickle.dump(non_zero_rows, f)

In [ ]:
print("END")